In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)

# Cache

Let's set up an in-memory cache and invoke an LLM:

In [3]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

cache = InMemoryCache()
set_llm_cache(cache)

llm.invoke("What is the capital of UK?")

AIMessage(content='The capital of the UK is **London**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019be7b7-e3e5-78d2-a443-c9ee00b02882-0', usage_metadata={'input_tokens': 8, 'output_tokens': 30, 'total_tokens': 38, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})

In [16]:
llm.invoke("What is the capital of UK?")

AIMessage(content='The capital of the UK is **London**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019be71f-58d0-7f72-ab76-69cccdcf1ecb-0', usage_metadata={'input_tokens': 8, 'output_tokens': 32, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}, 'total_cost': 0})

Now we can see the request-response pair has been cached:

In [ ]:
from langchain_core.globals import get_llm_cache
get_llm_cache()._cache

{('[{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "messages", "HumanMessage"], "kwargs": {"content": "What is the capital of UK?", "type": "human"}}]',
  '{"id": ["langchain_google_genai", "chat_models", "ChatGoogleGenerativeAI"], "kwargs": {"default_metadata": [], "google_api_key": {"id": ["GOOGLE_API_KEY"], "lc": 1, "type": "secret"}, "max_retries": 6, "model": "gemini-2.5-flash", "n": 1, "temperature": 1.0}, "lc": 1, "name": "ChatGoogleGenerativeAI", "type": "constructor"}---[(\'stop\', None)]'): [ChatGeneration(text='The capital of the UK is **London**.', generation_info={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, message=AIMessage(content='The capital of the UK is **London**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019be7b7-e3e5-78d2-a443-c9ee00b02882-0', usage_metadata={'input_tokens': 8, 'output

That's how an LLM instance is serialized for aching purposes:

In [15]:
llm._get_llm_string()

'{"id": ["langchain_google_genai", "chat_models", "ChatGoogleGenerativeAI"], "kwargs": {"default_metadata": [], "google_api_key": {"id": ["GOOGLE_API_KEY"], "lc": 1, "type": "secret"}, "max_retries": 6, "model": "gemini-2.5-flash", "n": 1, "temperature": 1.0}, "lc": 1, "name": "ChatGoogleGenerativeAI", "type": "constructor"}---[(\'stop\', None)]'

We can also create an in-memory store to store and search for key-value pairs:

In [16]:
from langgraph.store.memory import InMemoryStore
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

in_memory_store = InMemoryStore()

in_memory_store.put(namespace=("users", "user1"), key="fact1", value={"message1": "My name is John."})
in_memory_store.put(namespace=("users", "user1", "conv1"), key="address", value={"message": "I live in Berlin."})


In [17]:
in_memory_store.get(namespace=("users", "user1"), key="address")

In [18]:
in_memory_store.get(namespace=("users", "user1", "conv1"), key="address")

Item(namespace=['users', 'user1', 'conv1'], key='address', value={'message': 'I live in Berlin.'}, created_at='2026-01-22T22:05:01.404655+00:00', updated_at='2026-01-22T22:05:01.404656+00:00')

In [19]:
in_memory_store.search(("users", "user1", "conv1"), query="name")

[Item(namespace=['users', 'user1', 'conv1'], key='address', value={'message': 'I live in Berlin.'}, created_at='2026-01-22T22:05:01.404655+00:00', updated_at='2026-01-22T22:05:01.404656+00:00', score=None)]